In [15]:
#Does the average Body Mass Index (BMI) of players differ significantly between teams that advanced to the knockout stage (advanced) and teams that were eliminated in the group stage (eliminated) in the FIFA World Cup 2026?

#Data Sources & Documentation
#FB Ref (Primary Source for Player Physical Metrics):** https://fbref.com/en/
#FIFA Official Website (Tournament Statistics):** https://www.fifa.com/en/tournaments/mens/worldcup/canadamexicousa2026/statistics
#The Stats Don't Lie (Match & Team Outcomes):** https://www.thestatsdontlie.com/football/world-cup-2026/


#Import libraries
import pandas as pd
import numpy as np

In [9]:
#Loaded the dataset
df = pd.read_csv("FIFAWC2026_BMI_Cleaned.csv")

print(df.head())

        player_name    Pos  MP  Min  goals  height_cm  weight_kg   Team  \
0        Ãlex Baena  MF,FW   7  481      1        175         69  Spain   
1      Mikel Merino     MF   8  199      2        188         83  Spain   
2   Mikel Oyarzabal     FW   8  601      5        181         78  Spain   
3       Pedro Porro     DF   6  563      2        176         68  Spain   
4  Fabián Ruiz Peña     MF   8  321      1        189         69  Spain   

  team_status        BMI  
0    advanced  22.530612  
1    advanced  23.483477  
2    advanced  23.808797  
3    advanced  21.952479  
4    advanced  19.316369  


In [10]:
#Set target sample per group
target_sample_size = 50

#Count player in advanced group
n_advanced_pop = len(df[df['team_status'] == 'advanced'])

#Count player in eliminated group
n_eliminated_pop = len(df[df['team_status'] == 'eliminated'])

#Adjust sample size if population is smaller than the target
sample_size = min(target_sample_size, n_advanced_pop, n_eliminated_pop)

print("==================================================")
print("       SAMPLING VERIFICATION REPORT         ")
print("==================================================")
print(f"• Total Advanced Players in Population   : {n_advanced_pop}")
print(f"• Total Eliminated Players in Population : {n_eliminated_pop}")
print(f"• Target Sample Size Requested            : {target_sample_size} per group")
print("-" * 50)

#Check if the sampling group is correct
if sample_size < target_sample_size:
  print(f"[Warning] Limited data available. Adjusting sample size per group to: {sample_size}")
  print(f"   Adjusting sample size per group down to: {sample_size} (to match the smallest group)")
else:
  print(f" Sufficient data available in both groups!")
  print(f"   Sample size is safely set to: {sample_size} per group")
print("==================================================")


       SAMPLING VERIFICATION REPORT         
• Total Advanced Players in Population   : 706
• Total Eliminated Players in Population : 313
• Target Sample Size Requested            : 50 per group
--------------------------------------------------
 Sufficient data available in both groups!
   Sample size is safely set to: 50 per group


In [11]:
#Split the populations
df_advanced_pop = df[df['team_status'] == 'advanced']
df_eliminated_pop = df[df['team_status'] == 'eliminated']

#Simple Random Sampling
sample_advanced = df_advanced_pop.sample(n=sample_size, random_state=42)
sample_eliminated = df_eliminated_pop.sample(n=sample_size, random_state=42)

#Combine samples for descriptive stats analysis
df_sample = pd.concat([sample_advanced, sample_eliminated])

print(f"Simple Random Sampling executed successfully:")
print(f"  - Advanced sample group size (n) = {len(sample_advanced)}")
print(f"  - Eliminated sample group size (n) = {len(sample_eliminated)}")


Simple Random Sampling executed successfully:
  - Advanced sample group size (n) = 50
  - Eliminated sample group size (n) = 50


In [14]:
population_outliers = []

#Use the target population grouping variable: df_advanced_pop and df_eliminated_pop
for group_name, sub_pop in [('advanced', df_advanced_pop), ('eliminated', df_eliminated_pop)]:
    q1 = sub_pop['BMI'].quantile(0.25)
    q3 = sub_pop['BMI'].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - (1.5 * iqr)
    upper_bound = q3 + (1.5 * iqr)

    #Identify players with outlier BMI values
    outliers = sub_pop[(sub_pop['BMI'] < lower_bound) | (sub_pop['BMI'] > upper_bound)]
    print(f"\nGroup [{group_name.upper()}]:")
    print(f"  - IQR Statistics: Q1 = {q1:.4f}, Q3 = {q3:.4f}, IQR = {iqr:.4f}")
    print(f"  - Normal Range: [{lower_bound:.4f} to {upper_bound:.4f}]")
    print(f"  - Found {len(outliers)} population outlier(s)")

    #Show the names and body proportions of the outlier players
    if len(outliers) > 0:
        for idx, row in outliers.iterrows():
            population_outliers.append(row)
            print(f"    {row['player_name']} | BMI: {row['BMI']:.2f} (Ht: {row['height_cm']} cm, Wt: {row['weight_kg']} kg)")

print("-" * 69)


#Detect outliers at the random sample level
print("\n=====================================================================")
print("            SAMPLE BMI OUTLIER REPORT (GROUP-SPECIFIC IQR)           ")
print("=====================================================================")

sample_outliers = []

#Extract your random sample variable, df_sample, to calculate and check the IQR
sample_adv_group = df_sample[df_sample['team_status'] == 'advanced']
sample_elim_group = df_sample[df_sample['team_status'] == 'eliminated']

for group_name, sub_sample in [('advanced', sample_adv_group), ('eliminated', sample_elim_group)]:
    q1 = sub_sample['BMI'].quantile(0.25)
    q3 = sub_sample['BMI'].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - (1.5 * iqr)
    upper_bound = q3 + (1.5 * iqr)

    outliers_s = sub_sample[(sub_sample['BMI'] < lower_bound) | (sub_sample['BMI'] > upper_bound)]
    print(f"\nGroup [{group_name.upper()}]:")
    print(f"  - IQR Statistics: Q1 = {q1:.4f}, Q3 = {q3:.4f}, IQR = {iqr:.4f}")
    print(f"  - Normal Range: [{lower_bound:.4f} to {upper_bound:.4f}]")
    print(f"  - Found {len(outliers_s)} sample outlier(s)")

    if len(outliers_s) > 0:
        for idx, row in outliers_s.iterrows():
            sample_outliers.append(row)
            print(f"    {row['player_name']} | BMI: {row['BMI']:.2f} (Ht: {row['height_cm']} cm, Wt: {row['weight_kg']} kg)")

print("=====================================================================")


Group [ADVANCED]:
  - IQR Statistics: Q1 = 22.1016, Q3 = 23.8192, IQR = 1.7177
  - Normal Range: [19.5251 to 26.3957]
  - Found 26 population outlier(s)
    Fabián Ruiz Peña | BMI: 19.32 (Ht: 189 cm, Wt: 69 kg)
    ValentÃ­n Barco | BMI: 19.08 (Ht: 186 cm, Wt: 66 kg)
    Reece James | BMI: 27.72 (Ht: 172 cm, Wt: 82 kg)
    Ivan Toney | BMI: 27.15 (Ht: 179 cm, Wt: 87 kg)
    Azzedine Ounahi | BMI: 19.02 (Ht: 182 cm, Wt: 63 kg)
    Sofyan Amrabat | BMI: 27.73 (Ht: 173 cm, Wt: 83 kg)
    Noussair Mazraoui | BMI: 19.11 (Ht: 183 cm, Wt: 64 kg)
    Matias Fernandez-Pardo | BMI: 19.52 (Ht: 188 cm, Wt: 69 kg)
    Thelo Aasgaard | BMI: 26.99 (Ht: 170 cm, Wt: 78 kg)
    Leo Ã˜stigÃ¥rd | BMI: 26.75 (Ht: 174 cm, Wt: 81 kg)
    Sander Berge | BMI: 26.59 (Ht: 190 cm, Wt: 96 kg)
    Kristian Thorstvedt | BMI: 28.38 (Ht: 171 cm, Wt: 83 kg)
    Luca Jaquez | BMI: 27.74 (Ht: 187 cm, Wt: 97 kg)
    Gabriel Ãvalos | BMI: 26.59 (Ht: 185 cm, Wt: 91 kg)
    Rayan | BMI: 27.02 (Ht: 171 cm, Wt: 79 kg)
    Gu

In [6]:
df_sample.to_csv(
    "FIFAWC2026_Final_Sample.csv",
    index=False
)

print("Sample CSV saved successfully.")

Sample CSV saved successfully.
